# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계4 : 통합-모듈화**

## **0.미션**

단계 4에서는, 단계1,2,3 에서 생성한 함수들을 모듈화하고, 단위 테스트 및 파이프라인 코드를 작성합니다.

* **미션6**
    * Python 코드 모듈화
        * 각 모듈 코드 및 모델, 데이터파일을 일관성 있게 정리
        * .py 파일 생성 ==> 라이브러리 로딩, 각 task를 위한 함수 생성


## **1.환경설정**

* 경로 설정

구글 드라이브 연결

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/miniproj6/'

## 2.모듈 구성하기

In [ ]:
!pip install openai

In [ ]:
pip install haversine

In [ ]:
#%%writefile /content/drive/MyDrive/miniproj6/emergency.py

import os
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import openai
from openai import OpenAI
import json
import torch
import re
from haversine import haversine
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

#path = ''

# 0. load key file------------------
def load_file(filepath):
    with open(filepath, 'r') as file:
        return file.readline().strip()

# 1-1 audio2text--------------------
## Whisper API로 mp3파일 > 텍스트로 변환
def audio_to_text(audio_path, filename):
    openai_api_key = load_file(path + 'api_key.txt')
    # OpenAI 클라이언트 생성
    client = OpenAI(api_key=openai_api_key)
    # 음성파일 경로 지정
    audio_file = open(audio_path + filename, "rb")
    # 오디오 파일을 읽어서, 위스퍼를 사용한 변환
    transcript = client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-1",
        language="ko",
        response_format="text",
    )

    print('*'*50)
    print("STT RESULT: ",transcript)
    print('*'*50)
    # 결과 반환
    return transcript

# 1-2 text2summary------------------
## GPT로 텍스트에서 요약문과 리스트 추출
def summ_list_extract(user_message):
    openai_api_key = load_file(path + 'api_key.txt')
    # OpenAI 클라이언트 생성
    client = OpenAI(api_key=openai_api_key)
    #append if I say YES!
    messages = [
        {"role": "system", "content": "Act as a text summarizer, and word extractor. For any text given, your job is to summarize it into two sentecnes, and extract a word list from it."},
        {"role":"user","content": "아까 가다가 머리를 박았는데, 처음에는 괜찮다가 지금 세시간 정도 지났는데 머리가 어지럽고 속이 메스꺼워요 어떻게 해야할까요?"},
        {"role":"assistant","content": """#
가다가 머리를 박아 처음에는 괜찮았으나, 세시간 후부터 머리가 어지럽고 속이 메스꺼워진 상황
##
['머리', '박았다', '처음에', '괜찮다','세시간', '정도', '지났는데', '머리', '어지럽고', '속', '메스꺼워요', '어떻게', '해야할까요']"""},
    ]

    ##Generate response
    messages_with_user_input = messages + [{"role": "user", "content": user_message}]

    completion_response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages_with_user_input,
        max_tokens=2500
    )
    response_message = completion_response.choices[0].message.content

    print("#"*50)
    print("Bot: ", response_message)
    print("#"*50)

    ##Extract text
    summ_text = re.findall(r"#\n(.*)\n##", response_message, re.DOTALL)[0]
    print("Text: ", summ_text)
    print("#"*50)

    ##Extract list
    list_string = re.findall(r"\n##\n(\[.*\])", response_message, re.DOTALL)[0]
    list_data = eval(list_string)
    print("List: ", list_data)
    print("#"*50)

    return summ_text, list_data

# 2. model prediction------------------
# 데이터 예측 함수
def predict(text, model, tokenizer):
    # 입력 문장 토크나이징
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # 입력 텐서를 모델과 동일한 장치로 이동
    device = next(model.parameters()).device  # 모델의 현재 장치
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # 모델 예측
    with torch.no_grad():
        outputs = model(**inputs)

    # 로짓을 소프트맥스로 변환하여 확률 계산
    logits = outputs.logits
    probabilities = logits.softmax(dim=1)

    # 가장 높은 확률을 가진 클래스 선택
    pred = torch.argmax(probabilities, dim=-1).item()

    return pred, probabilities

# 3-1. get_distance------------------
def get_dist(start_lat, start_lng, dest_lat, dest_lng,):
    url = "https://naveropenapi.apigw.ntruss.com/map-direction/v1/driving"
    headers = {
        "X-NCP-APIGW-API-KEY-ID": 'nf30b2d7do',
        "X-NCP-APIGW-API-KEY": 'Ao1BZYROhTsZ72MeOCJgtAeoAZbqbzGFy2UXMo5h',
    }
    params = {
        "start": f"{start_lng},{start_lat}",  # 출발지 (경도, 위도)
        "goal": f"{dest_lng},{dest_lat}",    # 목적지 (경도, 위도)
        "option": "trafast"  # 실시간 빠른 길 옵션
    }

    # 요청하고, 답변 받아오기
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        response = response.json()
    else:
        raise Exception(f"API 요청 실패: {response.status_code} - {response.text}")
    #print(response)
#    dist = response['route']['trafast'][0]['summary']['duration']  # m(미터)
 #   dist = dist / 1000  # km로 변환

    duration = response['route']['trafast'][0]['summary']['duration']  # m(미터)

    return duration

# 3-2. recommendation------------------
def recommend_hospital(start_lat, start_lng): #a_lat a_lng는 0.2로해도됨

  df_emerg = pd.read_csv(path + 'er_loc.csv', index_col=0)
  df_emerg.reset_index(drop=True, inplace=True)

  a_lat = 0.2
  a_lng = 0.2
  coord = []
  df_emerg.reset_index(drop=True, inplace=True)
  df_emerg['harversine'] = np.NaN
  df_emerg['optimal path'] = np.NaN

  # a_lat, a_lng을 활용한 제한 설정
  df_emerg = df_emerg[(start_lat -  a_lat < df_emerg['위도']) &
   (df_emerg['위도'] < start_lat +  a_lat) &
   (start_lng -  a_lng < df_emerg['경도']) &
    (df_emerg['경도'] < start_lng +  a_lng)]
  df_emerg.reset_index(drop=True, inplace=True)

  # Haversine 거리
  for i in range(len(df_emerg)):
    dest_lat, dest_lng = df_emerg.loc[i, ['위도', '경도']]
    df_emerg.loc[i, 'harversine'] = haversine((start_lat, start_lng), (dest_lat, dest_lng), unit='km')
  df_emerg = df_emerg.sort_values(by='harversine')
  df_emerg = df_emerg.head(10)
  df_emerg.reset_index(drop=True, inplace=True)

  # 거리 계산
  for i in tqdm(range(10)):
    dest_lat, dest_lng = df_emerg.loc[i, ['위도', '경도']]
    df_emerg.loc[i, 'optimal path'] = get_dist(start_lat, start_lng, dest_lat, dest_lng)

  df_emerg = df_emerg.sort_values(by='optimal path')
  df_emerg.reset_index(drop=True, inplace=True)
  coord = [(i, j) for i, j in zip(df_emerg['위도'].iloc[:3], df_emerg['경도'].iloc[:3])]
  return coord

#print(get_dist(37.35,127.11,37.35,127.25))
print(recommend_hospital(37.35,127.11))
# a2t = audio_to_text(path,'audio1.mp3')
# print("Audio2Text",a2t)
# summ,list = summ_list_extract(a2t)
# print("SummedText",summ)
# # coord = recommend_hospital(start_lat, start_lng, df_emerg, 0.2, 0.4)
# # 모델,토크나이저 로드
# model = AutoModelForSequenceClassification.from_pretrained(path)
# tokenizer = AutoTokenizer.from_pretrained(path)
# predicted_class, probabilities = predict(summ, model, tokenizer)

# print(f"예측된 클래스 이름: {predicted_class+1}등급")
# print(f"클래스별 확률: {probabilities}")

 60%|██████    | 6/10 [00:06<00:04,  1.14s/it]